In [1]:
import os
import copy
import numpy as np
import pathlib
import rasterio
import geopandas as gpd
import pandas as pd

from dateutil.relativedelta import relativedelta
from openpyxl import load_workbook

from food_security import data_reader
from rasterio.transform import from_origin
from rasterio.mask import mask
import xarray as xr

In [2]:
src_dir = pathlib.Path("~").expanduser().resolve() / "data/food-security"
# src_dir = pathlib.Path('/Users/hemert/OneDrive - Stichting Deltares/Documents - International Delta Toolset/Agr production postprocessing/JCARWQV7.Rbd/1/')
ribasim_dir = src_dir / "Ribasim8_MekDelta/Modules/ribasim/Mekong.1/work/"

In [3]:
his_file = data_reader.HisFile(ribasim_dir / 'RIB_ADVIR_dmnd.his', crop=None)
# his_file = data_reader.HisFile(src_dir / "AirAgDmd.his", crop=None)
his_file.read(hia=True)
ds = his_file.ds

In [34]:
his_file = data_reader.HisFile(pathlib.Path("/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/JCARWQV7.Rbd/10/") / 'CLTAGPRO.HIS', crop=None)
# his_file = data_reader.HisFile(src_dir / "AirAgDmd.his", crop=None)
his_file.read(hia=False)
ds = his_file.ds

In [35]:
ds

<xarray.Dataset> Size: 427kB
Dimensions:                (time: 2, station: 1617)
Coordinates:
  * time                   (time) datetime64[us] 16B 2020-10-01 2021-10-02
  * station                (station) <U20 129kB 'Nd_______5 / Cr__1 /' ... 'N...
Data variables: (12/23)
    Demand (Mcm)           (time, station) float32 13kB 18.84 35.09 ... 14.55
    + Allocation (Mcm)     (time, station) float32 13kB 18.84 35.09 ... 5.053
    - Outflow (Mcm)        (time, station) float32 13kB 5.627 7.126 ... 0.2652
    Pot.evapotranspirat    (time, station) float32 13kB 13.21 27.97 ... 7.429
    - Act.evapotranspir    (time, station) float32 13kB 13.21 27.97 ... 7.429
    - Act.percolation (    (time, station) float32 13kB 0.0 0.0 0.0 ... 0.0 0.0
    ...                     ...
    Potent.production c    (time, station) float32 13kB 4.14e+07 ... 1.027e+08
    Actual field level     (time, station) float32 13kB 8.675e+06 ... 8.53e+07
    Actual farm gate pr    (time, station) float32 13kB 8.675e+06 ... 8.53e+07
    Actual field level_2   (time, station) float32 13kB 7.634e+07 ... 2.218e+08
    Actual farm gate pr_2  (time, station) float32 13kB 7.634e+07 ... 2.218e+08
    Actual production c    (time, station) float32 13kB 4.14e+07 ... 1.027e+08
Attributes:
    header:   All cultivations                        Yearly agriculture prod...
    scu:      86400
    t0:       2020-10-01 00:00:00

In [6]:
variable_names = list(ds.data_vars)

# Filter the list to only include variable names starting with 'P'
filtered_variable_names = [name for name in variable_names if name.startswith("P")]

In [7]:
def dt_to_str(df):
    return df.strftime("%m-%d")


def populate_crop_table(his_file_name, csv_file_name):
    df_dict = {
        "crop_id": [],
        "crop_name": [],
        "start_ts": [],
        "end_ts": [],
        "area_name": [],
    }

    his_file = data_reader.HisFile(his_file_name, crop=None)
    his_file.read(hia=True)
    ds = his_file.ds

    variable_names = list(ds.data_vars)
    potential_crop_names = [name for name in variable_names if name.startswith("P")]
    for potential_crop_name in potential_crop_names:
        crop_id_and_name = potential_crop_name.partition("P ")[2]
        crop_id = crop_id_and_name.split("/")[0]
        crop_name = crop_id_and_name.split("/")[1]

        for station in ds.station.values:
            ds_filtered = ds.sel({"station": station})[potential_crop_name]

            growing_times = ds_filtered.where(ds_filtered == 0, drop=True).time
            if growing_times.shape == (0,):
                start_of_year = str(ds_filtered.time.values.min())
            else:
                start_of_year = str(growing_times.values.min())
            year = int(start_of_year[:4])
            next_year = str(year + 1)
            end_of_year = next_year + start_of_year[4:]

            if (
                station == "5 / Blk_Air_20"
                and potential_crop_name == "P Cr06/Orchard_CITRU"
            ):
                5 == 5

            ds_filtered = ds_filtered.sel({"time": slice(start_of_year, end_of_year)})

            area_name_full = station.split(" / ")[1]
            area_name = area_name_full.split("_")[0]

            if ds_filtered.values.max() == 0:
                season_starts = [np.nan]
                season_ends = [np.nan]
            else:
                start = None
                season_starts = []
                season_ends = []
                for i, data in enumerate(ds_filtered):
                    if data.values != 0 and start is None:
                        start = i
                        season_starts.append(data.time.values)
                    if data.values == 0 and start is not None:
                        start = None
                        season_ends.append(ds_filtered.time.values[i - 1])
                if len(season_starts) > len(season_ends):
                    season_ends = copy.deepcopy(season_starts)

                for start_ts, end_ts in zip(season_starts, season_ends):
                    df_dict["crop_id"].append(crop_id)
                    df_dict["crop_name"].append(crop_name)
                    df_dict["start_ts"].append(start_ts)
                    df_dict["end_ts"].append(end_ts)
                    df_dict["area_name"].append(area_name)

    df = pd.DataFrame(df_dict)
    df = (
        df.groupby(["crop_id", "crop_name"])
        .agg({"start_ts": "min", "end_ts": "max"})
        .reset_index()
    )
    df["start_ts"] = df["start_ts"].apply(dt_to_str)
    df["end_ts"] = df["end_ts"].apply(dt_to_str)
    return df


def populate_area_table(his_file_name, csv_file_name):
    df_dict = {"area_id": [], "area_name": [], "area_map_name": []}

    his_file = data_reader.HisFile(his_file_name, crop=None)
    his_file.read()
    ds = his_file.ds

    for station in ds.station.values:
        area_id = station.split(" / ")[0]
        area_name_full = station.split(" / ")[1]
        area_name = area_name_full.split("_")[0]

        df_dict["area_id"].append(area_id)
        df_dict["area_name"].append(area_name)
        df_dict["area_map_name"].append(None)

    df = pd.DataFrame(df_dict)
    return df

In [8]:
df_crop = populate_crop_table(ribasim_dir / 'RIB_ADVIR_dmnd.his', None)
# df_crop = populate_crop_table(src_dir / "AirAgDmd.his", None)
df_crop_id = pd.DataFrame(
    {
        "crop_name": df_crop["crop_name"].unique(),
        "crop_name_fao": [None for i in range(len(df_crop["crop_name"].unique()))],
    }
)
df_area = populate_area_table(ribasim_dir / "RIB_ADVIR_dmnd.his", None)

In [9]:
ds.sel({"station": "5 / Blk_Air_20"})["P Cr06/Orchard_CITRU"].where(
    ds.sel({"station": "5 / Blk_Air_20"})["P Cr06/Orchard_CITRU"] == 0, drop=True
).time.shape == (0,)

KeyError: "not all values found in index 'station'. Try setting the `method` keyword argument (example: method='nearest')."

In [11]:
excel_path = src_dir / "input_test_egypt.xlsx"
write_dir = pathlib.Path("/Users/hemert/OneDrive - Stichting Deltares/Documents - International Delta Toolset/Salinity/Vietnam/")
excel_path = write_dir / 'input.xlsx'


# excel_path = '/Users/hemert/OneDrive - Stichting Deltares/Documents - International Delta Toolset/Salinity/Egypt/input.xlsx'
def write_excel_file(df, excel_path, sheet_name, append=False):
    # Open Excel file
    try:
        # book = load_workbook(excel_path)
        with pd.ExcelWriter(
            excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
        ) as writer:
            # excel_file.book = book
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    except Exception as e:
        print(e)
        df.to_excel(excel_path, sheet_name=sheet_name, index=False)

In [12]:
write_excel_file(df_crop_id, excel_path, "crop_id")
write_excel_file(df_crop, excel_path, "crop")
write_excel_file(df_area, excel_path, "area")

In [9]:
# -*- coding: utf-8 -*-
"""
Created on Thu Sep 21 11:36:06 2023

@author: vat
"""

from datetime import datetime
import os
import numpy as np
import struct as struct
import math as math
from datetime import datetime, timedelta

case = "18" #240619 v4: 2021, flood, HAD55.5
case = "19" #240619 v4: 2021, flood+lining, HAD55.5
case = "20" #240619 v4: 2021, drip+lining, HAD55.5
case = "21" #240619 v4: 2021, drip, HAD55.5
case = "22" #240619 v4: 2021, flood, HAD61
case = "23" #240619 v4: 2021, flood+lining, HAD61
case = "24" #240619 v4: 2021, drip, HAD61
case = "17" #240619 v4: 2021, drip+lining, HAD61

#function to read word number wordn from line number linen in file filename
def getvfrf(filename,linen,wordn):
    with open(filename,'rt') as file:
        for iline in range(linen):
            line=file.readline()
        result=line.split()[wordn-1]
        file.close()
        return result

# define class to read his files
class HisFile:
    def __init__(self, fname):
        self.fname = fname
        self.hianame = self.fname.with_suffix(".hia")
        # self.hianame = self.fname.replace(".his",".hia")

    def read(self):
        #try the hia file for long location names
        self.seghia = False
        try:
            f = open(self.hianame,'rt')
            for line in f:
                if line.strip() == "[Long Locations]":
                    self.seghia = True
                    self.longsegnums=list()
                    self.longsegnames=list()                
                    for line in f:
                        iseg=int(line.split("=")[0])
                        longname=line.split("=")[1].strip()
                        self.longsegnums.append(iseg)                 
                        self.longsegnames.append(longname)
                        #print(iseg,longname)
            f.close()
        except:
            self.seghia = False

        #try the hia file for long parameter names
        self.syshia = False
        try:
            f = open(self.hianame,'rt')
            for line in f:
                if line.strip() == "[Long Parameters]":
                    self.syshia = True
                    break
            if self.syshia:
                self.longsysnums=list()
                self.longsysnames=list()                
                for line in f:
                    if line.strip() == "":
                        break
                    else:
                        isys=int(line.split("=")[0])
                        longname=line.split("=")[1].strip()
                        self.longsysnums.append(isys)                 
                        self.longsysnames.append(longname)
                        #print(isys,longname)
            f.close()
        except:
            self.syshia = False

        #read the HIS file
        f = open(self.fname, 'rb')
        self.moname = f.read(160).decode()
        year = int(self.moname[124:128])
        month = int(self.moname[129:131])
        day = int(self.moname[132:134])
        hour = int(self.moname[135:137])
        minute = int(self.moname[138:140])
        second = int(self.moname[141:143])
        self.startdate = datetime(year, month, day, hour, minute, second)
        if self.moname[157] == 's':
            self.scu = int(self.moname[150:157])
        else:
            self.scu = int(self.moname[150:158])
        #print self.startdate,self.scu
        self.nsys, = struct.unpack('i', f.read(4))
        self.nseg, = struct.unpack('i', f.read(4))
        #print self.nsys, self.nseg
        self.sysnames = list()
        for isys in range(self.nsys):
            sysnam=f.read(20).decode()
            self.sysnames.append(str.rstrip(sysnam))
            #print(isys,sysnam,self.sysnames[isys])
        self.segnames = list()
        self.segnums = list()
        for iseg in range(self.nseg):
            self.segnums.append(struct.unpack('i', f.read(4))[0])
            segnam=f.read(20).decode()
            self.segnames.append(str.rstrip(segnam))
            #print (iseg,segnam,self.segnums[iseg],self.segnames[iseg])
        size = os.path.getsize(self.fname)
        self.nstep = int((size - 168 - 20 * self.nsys - 24 * self.nseg) / (4 * (self.nsys * self.nseg + 1))     )
        #print self.nstep
        self.datetime = list()
        self.data = np.zeros((self.nsys, self.nseg, self.nstep), 'f')
        for lstep in range(self.nstep):
            fdays = struct.unpack('i', f.read(4))[0] * (self.scu / 86400.)
            iday = math.trunc(fdays)
            ihour = math.trunc((fdays - iday) * 24.)
            iminute = math.trunc((fdays - iday - ihour / 24.) * 60.)
            isecond = fdays * 86400 - 60 * math.trunc(fdays * 86400 / 60.)
            dt = self.startdate + timedelta(iday, ihour, iminute, isecond)
            self.datetime.append(dt)
            for iseg in range(self.nseg):
                for isys in range(self.nsys):
                    self.data[isys, iseg, lstep] = struct.unpack('f', f.read(4))[0]

    def getseries(self, sysnam, segnam):
        if self.syshia:
            i = self.longsysnames.index(sysnam) 
            isys = self.longsysnums[i]
        else:
            isys = self.sysnames.index(sysnam)
        if self.seghia:
            i = self.longsegnames.index(segnam)
            iseg=self.longsegnums[i]
        else:
            iseg = self.segnames.index(segnam)
        #print isys,iseg
        res = np.zeros(self.nstep)
        for lstep in range(self.nstep):
            res[lstep] = self.data[isys, iseg, lstep]
        return res

    def gettimeseries(self, sysnam, segnam):
        if self.syshia:
            i = self.longsysnames.index(sysnam) 
            isys = self.longsysnums[i]
        else:
            isys = self.sysnames.index(sysnam)
        if self.seghia:
            i = self.longsegnames.index(segnam)
            iseg=self.longsegnums[i]
        else:
            iseg = self.segnames.index(segnam)
       #print isys,iseg
        resdata = np.zeros((self.nstep))
        resdate = list()
        for lstep in range(self.nstep):
            resdate.append(self.datetime[lstep])
            resdata[lstep] = self.data[isys, iseg, lstep]
        return resdate, resdata
    
    def gettimeaverage(self,sysnam,segnam):
        if self.syshia:
            i = self.longsysnames.index(sysnam) 
            isys = self.longsysnums[i]
        else:
            isys = self.sysnames.index(sysnam)
        if self.seghia:
            i = self.longsegnames.index(segnam)
            wqseg=self.longsegnums[i]
            wqsegnam="Segment" + str(wqseg).rjust(7)
            iseg = self.segnames.index(wqsegnam)
        else:
            iseg = self.segnames.index(segnam)
        #print isys,iseg
        total=0.
        for lstep in range(self.nstep):
            total += self.data[isys, iseg, lstep]
        res = total / self.nstep
        return res

#list of units
unitlst=[1,2,6,7,8,9,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,62,63,65,77,78] 

folder = 'JCARWQ.Rbd'


#input file names
if case =="WORK":
    descfolder = "CMTWORK"
else:
    descfolder = case
sumfile = 'c:\\Ribasim7\\{}\\{}\\summary.log'.format(folder, case)
sumfile = "/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/JCARWQV7.Rbd/10/summary.log"
balfile = 'c:\\Ribasim7\\{}\\{}\\ribalans.log'.format(folder, case)
balfile = "/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/JCARWQV7.Rbd/10/ribalans.log"
airfile = 'c:\\Ribasim7\\{}\\{}\\OvAlWtBl.prt'.format(folder, case)
airfile = "/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/JCARWQV7.Rbd/10/OvAlWtBl.prt"
hisfile = 'c:\\Ribasim7\\{}\\{}\\wq_rib.his'.format(folder, case)
hasWQ = os.path.isfile(hisfile)
if(hasWQ):
    wqrib=HisFile(hisfile)
    wqrib.read()
hisfile = 'c:\\Ribasim7\\{}\\{}\\CltAgPro.his'.format(folder, case)
hisfile = pathlib.Path("/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/JCARWQV7.Rbd/10/") / 'CLTAGPRO.HIS'
yieldhis=HisFile(hisfile)
yieldhis.read()
# casedesc= 'c:\\Ribasim7\\{}\\{}\\casedesc.cmt'.format(folder, descfolder)
# #create output file name
# respath = 'C:/Users/mulde_ns/OneDrive - Stichting Deltares/projects/JCAR-egypt/A4I-ribasim-taskfoce/Results'
# filetimesec=os.path.getmtime(balfile)
# filetime=datetime.fromtimestamp(filetimesec)
# resfile = respath + '\\RibasimResults' + filetime.strftime("%y%m%d-%H%M") + '.txt'

#get description from first two lines of casedesc.cmt
# with open(casedesc,'rt') as file:
#     line=file.readline()
#     sline = line.strip() + ' '
#     line=file.readline()
#     sline += line
#     file.close()

#get salinity data from the wq_rib.his file
# salaver=0.
# if(hasWQ):
#     sysnam="TDS"
#     count=0
#     for cu in unitlst:
#         if cu == 78:
#             #Toska
#             segnam= "Canal_361"
#         else:
#             segnam= "LFlow_PWS_Dom_{}".format(cu)
#         sal=wqrib.gettimeaverage(sysnam, segnam)
#         #print(cu,segnam,sal)
#         salaver += sal
#         count += 1

#     #calculatye the average
#     salaver = salaver / count    

#get wheat and rice production from CltAgPro.his
rice="Cr__7"
wheat="Cr_12"
rice_yield=0.
wheat_yield = 0.
sysnam = "Actual field level production [kg]"
for segnam in yieldhis.segnames:
    if rice in segnam:
        yieldval = yieldhis.getseries(sysnam, segnam)
        #print(segnam,yieldval)
        rice_yield += yieldval[0] * 1e-6
    if wheat in segnam:
        yieldval = yieldhis.getseries(sysnam, segnam)
        #print(segnam,yieldval)
        wheat_yield += yieldval[0] * 1e-6

#calculte system efficiency
HADrel = float(getvfrf(balfile,947,5))*0.001
irrisup=float(getvfrf(balfile,957,5))*0.001
pwssup = float(getvfrf(balfile,958,6))*0.001
fishsup = float(getvfrf(balfile,959,5))*0.001
irriret= float(getvfrf(balfile,949,6))*0.001
pwsret= float(getvfrf(balfile,950,7))*0.001
gwret= float(getvfrf(balfile,952,8))*0.001
gwstorincr = float(getvfrf(balfile,956,7))*0.001
gwstordecr = float(getvfrf(balfile,948,6))*0.001
syseff = (irrisup-irriret+pwssup-pwsret) / (HADrel)*100

sline = ''
#read numbers from RIBASIM output files and add them to sline
sline += '                          water balance      model\n'
sline += 'HAD release                  61.30           {:.2f}\n'.format(HADrel)
sline += 'Irrigation water demand      60.00           {:.2f}\n'.format(float(getvfrf(sumfile,168,2))*0.001)
sline += 'Field evaporation            40.00           {:.2f}\n'.format(float(getvfrf(airfile,30,2))*0.001)
sline += 'Irrigation water supply      60.43           {:.2f}\n'.format(irrisup)
sline += 'Re-use shallow groundwater    3.88           {:.2f}\n'.format(float(getvfrf(balfile,974,6))*0.001)
sline += 'Re-use drainage water        13.71           {:.2f}\n'.format(float(getvfrf(balfile,1032,3))*0.001)
sline += 'PWS water demand             12.36           {:.2f}\n'.format(float(getvfrf(sumfile,585,2))*0.001)
sline += 'PWS water supply             12.36           {:.2f}\n'.format(pwssup)
sline += 'Fish pond water demand        4.20           {:.2f}\n'.format(float(getvfrf(sumfile,453,2))*0.001)
sline += 'Fish pond water supply        4.20           {:.2f}\n'.format(fishsup)
sline += 'Groundwater storage decrease                 {:.2f}\n'.format(gwstordecr)
sline += 'Groundwater storage increase                 {:.2f}\n'.format(gwstorincr)
sline += 'Outflow                      13.25           {:.2f}\n'.format(float(getvfrf(balfile,955,4))*0.001)
# sline += 'Average canal salinity (ppm)   243           {:.0f}\n'.format(salaver)
sline += 'Wheat production (1000ton)   8,705           {:,.0f}\n'.format(wheat_yield)
sline += 'Rice production (1000ton)    4,107           {:,.0f}\n'.format(rice_yield)
sline += 'System efficiency (%)                        {:.0f}\n'.format(syseff)

#output to the screen
print(sline)

# #outpout to file
# with open(resfile,'wt') as file:
#     file.write(sline)
#     file.close()

                          water balance      model
HAD release                  61.30           61.81
Irrigation water demand      60.00           55.77
Field evaporation            40.00           37.84
Irrigation water supply      60.43           54.13
Re-use shallow groundwater    3.88           2.75
Re-use drainage water        13.71           7.35
PWS water demand             12.36           11.11
PWS water supply             12.36           11.11
Fish pond water demand        4.20           4.70
Fish pond water supply        4.20           2.50
Groundwater storage decrease                 0.14
Groundwater storage increase                 1.57
Outflow                      13.25           12.87
Wheat production (1000ton)   8,705           8,789
Rice production (1000ton)    4,107           4,273
System efficiency (%)                        80



In [11]:
yieldhis.getseries(sysnam, segnam)

array([85300144., 85300144.])

In [27]:
def getseries(self, sysnam, segnam):
    if self.syshia:
        i = self.longsysnames.index(sysnam) 
        isys = self.longsysnums[i]
    else:
        isys = self.sysnames.index(sysnam)
    if self.seghia:
        i = self.longsegnames.index(segnam)
        iseg=self.longsegnums[i]
    else:
        iseg = self.segnames.index(segnam)
    #print isys,iseg
    res = np.zeros(self.nstep)
    for lstep in range(self.nstep):
        res[lstep] = self.data[isys, iseg, lstep]

yieldhis.longsysnames[14]
yieldhis.segnames[1616]

'Nd____1147 / Cr_31 /'

In [33]:
yieldhis.data[14, 1616, 0]

np.float32(8.530015e+07)

In [42]:
ds.sel({'station': yieldhis.segnames[1616]})['Actual farm gate pr']

<xarray.DataArray 'Actual farm gate pr' (time: 2)> Size: 8B
array([8.530014e+07, 8.530014e+07], dtype=float32)
Coordinates:
  * time     (time) datetime64[us] 16B 2020-10-01 2021-10-02
    station  <U20 80B 'Nd____1147 / Cr_31 /'

In [45]:
ds.sel({'time': '2020-10-01'})['Actual farm gate pr']#.sum(dim='station')

<xarray.DataArray 'Actual farm gate pr' (station: 1617)> Size: 6kB
array([8.6753640e+06, 3.0501837e+08, 3.7222252e+07, ..., 1.4350189e+06,
       8.1328810e+06, 8.5300144e+07], shape=(1617,), dtype=float32)
Coordinates:
  * station  (station) <U20 129kB 'Nd_______5 / Cr__1 /' ... 'Nd____1147 / Cr...
    time     datetime64[us] 8B 2020-10-01

In [49]:
mask = ds.station.str.contains(wheat, regex=False)
ds_sel = ds.sel(station=mask, time='2020-10-01')
ds_sel['Actual farm gate pr'].sum()

<xarray.DataArray 'Actual farm gate pr' ()> Size: 4B
array(8.789321e+09, dtype=float32)
Coordinates:
    time     datetime64[us] 8B 2020-10-01

In [50]:
ds_sel.station

<xarray.DataArray 'station' (station: 66)> Size: 5kB
array(['Nd_______5 / Cr_12 /', 'Nd_______9 / Cr_12 /', 'Nd______10 / Cr_12 /',
       'Nd______16 / Cr_12 /', 'Nd______14 / Cr_12 /', 'Nd______25 / Cr_12 /',
       'Nd______35 / Cr_12 /', 'Nd______44 / Cr_12 /', 'Nd______47 / Cr_12 /',
       'Nd______19 / Cr_12 /', 'Nd______28 / Cr_12 /', 'Nd______27 / Cr_12 /',
       'Nd______33 / Cr_12 /', 'Nd______61 / Cr_12 /', 'Nd______64 / Cr_12 /',
       'Nd______41 / Cr_12 /', 'Nd______81 / Cr_12 /', 'Nd______92 / Cr_12 /',
       'Nd______91 / Cr_12 /', 'Nd______89 / Cr_12 /', 'Nd______96 / Cr_12 /',
       'Nd______42 / Cr_12 /', 'Nd______51 / Cr_12 /', 'Nd_____108 / Cr_12 /',
       'Nd_____120 / Cr_12 /', 'Nd_____124 / Cr_12 /', 'Nd_____128 / Cr_12 /',
       'Nd_____132 / Cr_12 /', 'Nd_____133 / Cr_12 /', 'Nd_____130 / Cr_12 /',
       'Nd_____126 / Cr_12 /', 'Nd_____147 / Cr_12 /', 'Nd_____148 / Cr_12 /',
       'Nd_____156 / Cr_12 /', 'Nd_____168 / Cr_12 /', 'Nd_____169 / Cr_12 /',
       'Nd_____154 / Cr_12 /', 'Nd_____175 / Cr_12 /', 'Nd_____166 / Cr_12 /',
       'Nd_____172 / Cr_12 /', 'Nd_____173 / Cr_12 /', 'Nd_____195 / Cr_12 /',
       'Nd_____200 / Cr_12 /', 'Nd_____204 / Cr_12 /', 'Nd_____197 / Cr_12 /',
       'Nd_____205 / Cr_12 /', 'Nd_____212 / Cr_12 /', 'Nd_____211 / Cr_12 /',
       'Nd_____201 / Cr_12 /', 'Nd_____224 / Cr_12 /', 'Nd_____222 / Cr_12 /',
       'Nd_____226 / Cr_12 /', 'Nd_____215 / Cr_12 /', 'Nd_____216 / Cr_12 /',
       'Nd_____220 / Cr_12 /', 'Nd_____283 / Cr_12 /', 'Nd_____213 / Cr_12 /',
       'Nd_____153 / Cr_12 /', 'Nd_____158 / Cr_12 /', 'Nd_____164 / Cr_12 /',
       'Nd______62 / Cr_12 /', 'Nd_____233 / Cr_12 /', 'Nd____1136 / Cr_12 /',
       'Nd____1140 / Cr_12 /', 'Nd____1143 / Cr_12 /', 'Nd____1147 / Cr_12 /'],
      dtype='<U20')
Coordinates:
  * station  (station) <U20 5kB 'Nd_______5 / Cr_12 /' ... 'Nd____1147 / Cr_1...
    time     datetime64[us] 8B 2020-10-01